# LINZ Data Download Examples

This notebook demonstrates practical examples of downloading LINZ imagery data using the batch download functionality.

**Learning objectives:**
- 🎯 Simple downloads with convenience functions
- ⚙️ Advanced downloads with custom parameters
- 📊 Understanding download progress and error handling
- 🔧 Setting up different configurations for various use cases

## Prerequisites

Make sure you have:
```bash
pip install obstore
```

And that the `imagery_aws_read.py` script is available in your workspace.

## 1. Import Functions and Check Availability

In [ ]:
# Import the functions from our imagery_aws_read module
try:
    from imagery_aws_read import (
        download_dataset_images, 
        get_public_store, 
        download_all_images, 
        NZ_DATASETS
    )
    print("✅ Successfully imported NZ data access functions!")
except ImportError as e:
    print(f"❌ Error importing functions: {e}")
    print("💡 Make sure the imagery_aws_read.py file is in your current directory.")
    print("💡 You can also copy the functions from the main script.")

✅ Successfully imported LINZ data access functions!


## 2. Check Available Datasets

In [ ]:
def list_available_datasets():
    """Show available datasets."""
    print("🗺️  === Available NZ Datasets ===")
    for name, info in NZ_DATASETS.items():
        print(f"   📊 {name}: {info.bucket} ({info.region})")
    
    print("\n📝 Dataset descriptions:")
    print("   • imagery: Aerial photography and satellite imagery")
    print("   • elevation: Digital elevation models and terrain data")
    print("   • coastal: Coastal and marine datasets")

list_available_datasets()

🗺️  === Available LINZ Datasets ===
   📊 imagery: nz-imagery (ap-southeast-2)
   📊 elevation: nz-elevation (ap-southeast-2)
   📊 coastal: nz-coastal (ap-southeast-2)

📝 Dataset descriptions:
   • imagery: Aerial photography and satellite imagery
   • elevation: Digital elevation models and terrain data
   • coastal: Coastal and marine datasets


## 3. Example 1: Simple Download Using Convenience Function

In [4]:
def example_simple_download():
    """Simple example using the convenience function."""
    print("🎯 === Simple Download Example ===")
    print("Downloading up to 5 images from Taranaki dataset...")
    
    try:
        # Download up to 5 images from Taranaki dataset to 'sample_images' directory
        files = download_dataset_images(
            dataset_name="imagery",
            path_prefix="taranaki/taranaki_2022-2023_0.1m/rgb/2193/",
            output_dir="sample_images",
            limit=5
        )
        
        print(f"\n✅ Downloaded {len(files)} files to sample_images/")
        
        if files:
            print("\n📄 Downloaded files:")
            for i, file_path in enumerate(files, 1):
                filename = file_path.split('/')[-1] if '/' in file_path else file_path.split('\\')[-1]
                print(f"   {i}. {filename}")
        
        return files
        
    except Exception as e:
        print(f"❌ Error in simple download: {e}")
        print("💡 Check your internet connection and obstore installation.")
        return []

# Run the simple example
simple_files = example_simple_download()

🎯 === Simple Download Example ===
Scanning for images in: taranaki/taranaki_2022-2023_0.1m/rgb/2193/
Found 2 image files to download
[1/2] Downloading: BH28_500_095032.tiff
  ✓ Downloaded 9.73 MB in 0.5s (18.04 MB/s)
[2/2] Downloading: BH28_500_095033.tiff
  ✓ Downloaded 11.92 MB in 0.3s (34.95 MB/s)

Download Summary:
  Successfully downloaded: 2 files
  Failed downloads: 0 files

✅ Downloaded 2 files to sample_images/

📄 Downloaded files:
   1. BH28_500_095032.tiff
   2. BH28_500_095033.tiff


## 4. Example 2: Custom Download with Manual Store Setup

In [ ]:
def example_custom_download():
    """More advanced example with custom parameters."""
    print("\n⚙️  === Custom Download Example ===")
    print("Setting up manual store configuration for more control...")
    
    try:
        # Set up store manually for more control
        dataset = NZ_DATASETS["imagery"]
        store = get_public_store(bucket=dataset.bucket, region=dataset.region)
        
        print(f"🔌 Connected to {dataset.bucket} in {dataset.region}")
        
        # Download only TIFF files with custom settings
        print("📥 Downloading only TIFF files from Manawatu-Whanganui region...")
        
        files = download_all_images(
            store=store,
            prefix="manawatu-whanganui/",
            output_dir="tiff_downloads",
            image_extensions=(".tif", ".tiff"),  # Only TIFF files
            limit=3  # Limit to 3 files for demo
        )
        
        print(f"\n✅ Downloaded {len(files)} TIFF files to tiff_downloads/")
        
        if files:
            print("\n📊 File details:")
            for i, file_path in enumerate(files, 1):
                import os
                filename = os.path.basename(file_path)
                try:
                    size_mb = os.path.getsize(file_path) / (1024 * 1024)
                    print(f"   {i}. {filename} ({size_mb:.2f} MB)")
                except:
                    print(f"   {i}. {filename} (size unknown)")
        
        return files
        
    except Exception as e:
        print(f"❌ Error in custom download: {e}")
        print("💡 The prefix might not exist or might not contain TIFF files.")
        return []

# Run the custom example
custom_files = example_custom_download()


⚙️  === Custom Download Example ===
Setting up manual store configuration for more control...
🔌 Connected to nz-imagery in ap-southeast-2
📥 Downloading only TIFF files from Manawatu-Whanganui region...
Scanning for images in: manawatu-whanganui/
Found 1 image files to download
[1/1] Downloading: BN33_1000_1129.tiff
  ✓ Downloaded 1.99 MB in 0.5s (4.24 MB/s)

Download Summary:
  Successfully downloaded: 1 files
  Failed downloads: 0 files

✅ Downloaded 1 TIFF files to tiff_downloads/

📊 File details:
   1. BN33_1000_1129.tiff (1.99 MB)


## 5. Example 3: Different File Types and Filters

In [7]:
def example_file_type_filtering():
    """Demonstrate filtering by different file types."""
    print("\n🔍 === File Type Filtering Example ===")
    
    # Test different file extension combinations
    filter_examples = [
        {
            "name": "TIFF files only",
            "extensions": (".tif", ".tiff"),
            "output_dir": "tiff_downloads"
        },
        {
            "name": "JSON files only", 
            "extensions": (".json",),
            "output_dir": "json_downloads"
        }
    ]
    
    results = {}
    
    for example in filter_examples:
        print(f"\n📂 Testing: {example['name']}")
        print(f"   Extensions: {example['extensions']}")
        
        try:
            files = download_dataset_images(
                dataset_name="imagery",
                path_prefix="taranaki/taranaki_2022-2023_0.1m/rgb/2193/", 
                output_dir=example['output_dir'],
                limit=2  # Just 2 files per test
            )
            
            # Filter by our desired extensions (post-processing filter)
            filtered_files = [
                f for f in files 
                if any(f.lower().endswith(ext) for ext in example['extensions'])
            ]
            
            results[example['name']] = filtered_files
            print(f"   ✅ Found {len(filtered_files)} matching files")
            
        except Exception as e:
            print(f"   ❌ Error: {e}")
            results[example['name']] = []
    
    # Summary
    print(f"\n📊 === Filter Results Summary ===")
    for name, files in results.items():
        print(f"   {name}: {len(files)} files")
    
    return results

# Run the filtering example
filter_results = example_file_type_filtering()


🔍 === File Type Filtering Example ===

📂 Testing: TIFF files only
   Extensions: ('.tif', '.tiff')
Scanning for images in: taranaki/taranaki_2022-2023_0.1m/rgb/2193/
Found 1 image files to download
[1/1] Downloading: BH28_500_095032.tiff
  ✓ Downloaded 9.73 MB in 1.0s (10.00 MB/s)

Download Summary:
  Successfully downloaded: 1 files
  Failed downloads: 0 files
   ✅ Found 1 matching files

📂 Testing: JSON files only
   Extensions: ('.json',)
Scanning for images in: taranaki/taranaki_2022-2023_0.1m/rgb/2193/
Found 1 image files to download
[1/1] Downloading: BH28_500_095032.tiff
  ✓ Downloaded 9.73 MB in 0.9s (10.31 MB/s)

Download Summary:
  Successfully downloaded: 1 files
  Failed downloads: 0 files
   ✅ Found 0 matching files

📊 === Filter Results Summary ===
   TIFF files only: 1 files
   JSON files only: 0 files


## 6. Example 4: Testing Different Regions and Limits

In [8]:
def example_different_regions():
    """Test downloads from different regions with various limits."""
    print("\n🌍 === Different Regions Example ===")
    
    # Different region prefixes to test
    region_tests = [
        {
            "name": "Taranaki (Small limit)",
            "prefix": "taranaki/",
            "limit": 1,
            "output_dir": "taranaki_sample"
        },
        {
            "name": "Canterbury (Medium limit)",
            "prefix": "canterbury/",
            "limit": 2,
            "output_dir": "canterbury_sample"
        },
        {
            "name": "Auckland (Small limit)",
            "prefix": "auckland/",
            "limit": 1,
            "output_dir": "auckland_sample"
        }
    ]
    
    region_results = {}
    
    for region in region_tests:
        print(f"\n🏞️  Testing region: {region['name']}")
        print(f"   Prefix: {region['prefix']}")
        print(f"   Limit: {region['limit']} files")
        
        try:
            files = download_dataset_images(
                dataset_name="imagery",
                path_prefix=region['prefix'],
                output_dir=region['output_dir'],
                limit=region['limit']
            )
            
            region_results[region['name']] = {
                'files': files,
                'count': len(files),
                'success': True
            }
            
            print(f"   ✅ Successfully downloaded {len(files)} files")
            
        except Exception as e:
            print(f"   ❌ Error downloading from {region['name']}: {e}")
            region_results[region['name']] = {
                'files': [],
                'count': 0,
                'success': False,
                'error': str(e)
            }
    
    # Results summary
    print(f"\n📈 === Regional Download Summary ===")
    total_files = 0
    successful_regions = 0
    
    for region_name, result in region_results.items():
        if result['success']:
            print(f"   ✅ {region_name}: {result['count']} files")
            total_files += result['count']
            successful_regions += 1
        else:
            print(f"   ❌ {region_name}: Failed")
    
    print(f"\n📊 Overall: {successful_regions}/{len(region_tests)} regions, {total_files} total files")
    
    return region_results

# Run the regional example
regional_results = example_different_regions()


🌍 === Different Regions Example ===

🏞️  Testing region: Taranaki (Small limit)
   Prefix: taranaki/
   Limit: 1 files
Scanning for images in: taranaki/
No image files found.
   ✅ Successfully downloaded 0 files

🏞️  Testing region: Canterbury (Medium limit)
   Prefix: canterbury/
   Limit: 2 files
Scanning for images in: canterbury/
Found 1 image files to download
[1/1] Downloading: BX18_500_054089.tiff
  ✓ Downloaded 0.38 MB in 0.4s (0.94 MB/s)

Download Summary:
  Successfully downloaded: 1 files
  Failed downloads: 0 files
   ✅ Successfully downloaded 1 files

🏞️  Testing region: Auckland (Small limit)
   Prefix: auckland/
   Limit: 1 files
Scanning for images in: auckland/
No image files found.
   ✅ Successfully downloaded 0 files

📈 === Regional Download Summary ===
   ✅ Taranaki (Small limit): 0 files
   ✅ Canterbury (Medium limit): 1 files
   ✅ Auckland (Small limit): 0 files

📊 Overall: 3/3 regions, 1 total files


## 7. Error Handling and Troubleshooting Example

In [ ]:
def example_error_handling():
    """Demonstrate error handling and troubleshooting."""
    print("\n🛠️  === Error Handling Example ===")
    
    # Test cases that might fail
    error_test_cases = [
        {
            "name": "Invalid dataset",
            "test": lambda: download_dataset_images(
                dataset_name="nonexistent",  # This should fail
                path_prefix="test/",
                limit=1
            )
        },
        {
            "name": "Empty prefix",
            "test": lambda: download_dataset_images(
                dataset_name="imagery",
                path_prefix="definitely/does/not/exist/",  # Likely empty
                limit=1
            )
        },
        {
            "name": "Valid small download",
            "test": lambda: download_dataset_images(
                dataset_name="imagery",
                path_prefix="taranaki/taranaki_2022-2023_0.1m/rgb/2193/",
                limit=1
            )
        }
    ]
    
    results = []
    
    for test_case in error_test_cases:
        print(f"\n🧪 Testing: {test_case['name']}")
        
        try:
            files = test_case['test']()
            
            if files:
                print(f"   ✅ Success: Downloaded {len(files)} files")
                result = {'name': test_case['name'], 'status': 'success', 'count': len(files)}
            else:
                print(f"   ⚠️  No files found (empty result)")
                result = {'name': test_case['name'], 'status': 'empty', 'count': 0}
                
        except ValueError as e:
            print(f"   ❌ ValueError: {e}")
            result = {'name': test_case['name'], 'status': 'value_error', 'error': str(e)}
            
        except Exception as e:
            print(f"   ❌ Unexpected error: {e}")
            result = {'name': test_case['name'], 'status': 'error', 'error': str(e)}
            
        results.append(result)
    
    # Summary of error handling
    print(f"\n🎯 === Error Handling Summary ===")
    for result in results:
        status_emoji = {
            'success': '✅',
            'empty': '⚠️',
            'value_error': '❌',
            'error': '💥'
        }.get(result['status'], '❓')
        
        print(f"   {status_emoji} {result['name']}: {result['status']}")
    
    return results

# Run error handling examples
error_results = example_error_handling()

## 8. Performance and Progress Monitoring

In [ ]:
import time\n",
    "import os\n",
    "\n",
    "def example_performance_monitoring():\n",
    "    \"\"\"Monitor download performance and progress.\"\"\"\n",
    "    print(\"\\n⏱️  === Performance Monitoring Example ===\")\n",
    "    \n",
    "    # Test different download sizes\n",
    "    performance_tests = [\n",
    "        {'limit': 1, 'name': 'Single file'},\n",
    "        {'limit': 3, 'name': 'Small batch'},\n",
    "        {'limit': 5, 'name': 'Medium batch'}\n",
    "    ]\n",
    "    \n",
    "    performance_results = []\n",
    "    \n",
    "    for test in performance_tests:\n",
    "        print(f\"\\n📊 Testing: {test['name']} ({test['limit']} files)\")\n",
    "        \n",
    "        start_time = time.time()\n",
    "        \n",
    "        try:\n",
    "            files = download_dataset_images(\n",
    "                dataset_name=\"imagery\",\n",
    "                path_prefix=\"taranaki/taranaki_2022-2023_0.1m/rgb/2193/\",\n",
    "                output_dir=f\"performance_test_{test['limit']}\",\n",
    "                limit=test['limit']\n",
    "            )\n",
    "            \n",
    "            end_time = time.time()\n",
    "            duration = end_time - start_time\n",
    "            \n",
    "            # Calculate total size\n",
    "            total_size = 0\n",
    "            for file_path in files:\n",
    "                try:\n",
    "                    total_size += os.path.getsize(file_path)\n",
    "                except:\n",
    "                    pass\n",
    "            \n",
    "            total_size_mb = total_size / (1024 * 1024)\n",
    "            speed_mbps = total_size_mb / duration if duration > 0 else 0\n",
    "            \n",
    "            result = {\n",
    "                'name': test['name'],\n",
    "                'files': len(files),\n",
    "                'duration': duration,\n",
    "                'size_mb': total_size_mb,\n",
    "                'speed_mbps': speed_mbps,\n",
    "                'success': True\n",
    "            }\n",
    "            \n",
    "            print(f\"   ✅ Downloaded {len(files)} files\")\n",
    "            print(f\"   ⏱️  Duration: {duration:.2f} seconds\")\n",
    "            print(f\"   📏 Total size: {total_size_mb:.2f} MB\")\n",
    "            print(f\"   🚀 Speed: {speed_mbps:.2f} MB/s\")\n",
    "            \n",
    "        except Exception as e:\n",
    "            duration = time.time() - start_time\n",
    "            result = {\n",
    "                'name': test['name'],\n",
    "                'files': 0,\n",
    "                'duration': duration,\n",
    "                'size_mb': 0,\n",
    "                'speed_mbps': 0,\n",
    "                'success': False,\n",
    "                'error': str(e)\n",
    "            }\n",
    "            print(f\"   ❌ Failed after {duration:.2f}s: {e}\")\n",
    "        \n",
    "        performance_results.append(result)\n",
    "    \n",
    "    # Performance summary\n",
    "    print(f\"\\n📈 === Performance Summary ===")\n",
    "    print(f\"{'Test':<15} {'Files':<6} {'Time (s)':<10} {'Size (MB)':<12} {'Speed (MB/s)':<15}\")\n",
    "    print(\"-\" * 65)\n",
    "    \n",
    "    for result in performance_results:\n",
    "        if result['success']:\n",
    "            print(f\"{result['name']:<15} {result['files']:<6} {result['duration']:<10.2f} {result['size_mb']:<12.2f} {result['speed_mbps']:<15.2f}\")\n",
    "        else:\n",
    "            print(f\"{result['name']:<15} {'FAIL':<6} {result['duration']:<10.2f} {'N/A':<12} {'N/A':<15}\")\n",
    "    \n",
    "    return performance_results\n",
    "\n",
    "# Run performance monitoring\n",
    "performance_data = example_performance_monitoring()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 9. Best Practices Summary"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def show_best_practices():\n",
    "    \"\"\"Display best practices for using the download functions.\"\"\"\n",
    "    print(\"🎯 === Best Practices for NZ Data Downloads ===")\n",
    "    \n",
    "    practices = [\n",
    "        {\n",
    "            'category': '🚀 Performance',\n",
    "            'tips': [\n",
    "                'Start with small limits (1-5 files) to test prefixes',\n",
    "                'Use appropriate file extensions to filter early',\n",
    "                'Consider network speed when setting batch sizes',\n",
    "                'Monitor download speeds and adjust accordingly'\n",
    "            ]\n",
    "        },\n",
    "        {\n",
    "            'category': '🛡️  Error Handling',\n",
    "            'tips': [\n",
    "                'Always wrap downloads in try-except blocks',\n",
    "                'Check if files already exist before downloading',\n",
    "                'Verify dataset names and prefixes before large downloads',\n",
    "                'Handle network interruptions gracefully'\n",
    "            ]\n",
    "        },\n",
    "        {\n",
    "            'category': '📁 File Management',\n",
    "            'tips': [\n",
    "                'Use descriptive output directory names',\n",
    "                'Organize downloads by region, date, or purpose', \n",
    "                'Check available disk space before large downloads',\n",
    "                'Clean up test downloads to save space'\n",
    "            ]\n",
    "        },\n",
    "        {\n",
    "            'category': '🔍 Exploration',\n",
    "            'tips': [\n",
    "                'Use list_objects() to explore available data first',\n",
    "                'Start with overview data before full resolution',\n",
    "                'Test different regions to understand data coverage',\n",
    "                'Check file sizes and formats before bulk downloads'\n",
    "            ]\n",
    "        }\n",
    "    ]\n",
    "    \n",
    "    for practice in practices:\n",
    "        print(f\"\\n{practice['category']}\")\n",
    "        for tip in practice['tips']:\n",
    "            print(f\"   • {tip}\")\n",
    "    \n",
    "    print(f\"\\n💡 Remember: NZ datasets are large and publicly funded resources.\")\n",
    "    print(f\"   Please use them responsibly and only download what you need!\")\n",
    "\n",
    "show_best_practices()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Summary\n",
    "\n",
    "This notebook demonstrated practical examples of downloading NZ data:\n",
    "\n",
    "✅ **Simple downloads** using convenience functions  \n",
    "✅ **Advanced downloads** with manual configuration  \n",
    "✅ **File type filtering** for specific formats  \n",
    "✅ **Regional testing** across different areas  \n",
    "✅ **Error handling** and troubleshooting  \n",
    "✅ **Performance monitoring** and optimization  \n",
    "\n",
    "### Key Takeaways:\n",
    "\n",
    "🎯 **Start Small**: Test with limits before large downloads  \n",
    "🛡️ **Handle Errors**: Always wrap downloads in error handling  \n",
    "📊 **Monitor Progress**: Track performance and file sizes  \n",
    "🔍 **Explore First**: Use listing functions to understand data structure  \n",
    "💾 **Manage Storage**: Organize downloads and clean up test files  \n",
    "\n",
    "### Next Steps:\n",
    "\n",
    "- Integrate downloads into your analysis workflow\n",
    "- Combine with rasterio for direct data processing\n",
    "- Build automated pipelines for regular data updates\n",
    "- Explore different NZ datasets (elevation, coastal)\n",
    "- Create custom processing functions for your specific needs"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.12.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 2
}